# Create Python 3.11 ENV

In [ ]:
!conda create -n femr-py311 python=3.11 -y
!conda run -n femr-py311 python -m pip install ipykernel
!conda run -n femr-py311 python -m pip install torch==2.1.2 \
    --index-url https://download.pytorch.org/whl/cu121
!conda run -n femr-py311 python -m ipykernel install \
    --user \
    --name femr-py311 \
    --display-name "Python 3.11 - FEMR"

In [ ]:
!conda run -n femr-py311 python -m pip install femr==0.2.3 datasets==2.15.0 xformers transformers==4.35.2


# Imports

In [ ]:
import os
import re
import pandas as pd
import numpy as np

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))

bucket = os.getenv("WORKSPACE_BUCKET")
cdr = os.environ.get("WORKSPACE_CDR")

if cdr is None:
    raise EnvironmentError(
        "WORKSPACE_CDR is not set. This script should be run inside an All of Us workspace."
    )

use_bqstorage = ("BIGQUERY_STORAGE_API_ENABLED" in os.environ)

# Load Model

In [ ]:
!conda run -n femr-py311 python -m pip install ipywidgets
from huggingface_hub import notebook_login
import femr.models.transformer
import torch
import femr.models.tokenizer
import femr.models.processor
import datetime

notebook_login()

In [ ]:
model_name = "StanfordShahLab/clmbr-t-base"

# Load tokenizer / batch loader
clmbr_tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
clmbr_batch_processor = femr.models.processor.FEMRBatchProcessor(clmbr_tokenizer)

# Load model
clmbr_model = femr.models.transformer.FEMRModel.from_pretrained(model_name)

In [ ]:
import inspect
import femr.models.transformer as femr_transformer

print(inspect.signature(
    femr_transformer.FEMREncoderLayer.__init__
))

print(inspect.signature(
    femr_transformer.FEMREncoderLayer.forward
))

print(inspect.signature(
    femr_transformer.FEMRTransformer.forward
))

print(inspect.getsource(
    femr.models.transformer.FEMREncoderLayer
))

print(inspect.getsource(
    femr_transformer.FEMRTransformer
))

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
model = AutoModelForMaskedLM.from_pretrained("boltuix/bert-mini")
tokenizer = AutoTokenizer.from_pretrained("boltuix/bert-mini")

## Full Model Pipeline

In [ ]:
import torch
import torch.nn as nn

import torch
import torch.nn as nn
import torch.nn.functional as F
import xformers

from torch.nn.utils.rnn import pad_sequence

import femr.models.transformer
from femr.models.transformer import fixed_pos_embedding

### Survey Cross attn

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence


class SurveyCrossAttn(nn.Module):
    def __init__(
        self,
        attn_heads,
        survey_dim,
        ehr_dim,
        dropout=0.1,
    ):
        super().__init__()

        if ehr_dim % attn_heads != 0:
            raise ValueError(
                f"ehr_dim={ehr_dim} must be divisible by "
                f"attn_heads={attn_heads}"
            )

        self.survey_embed_proj = (
            nn.Identity()
            if survey_dim == ehr_dim
            else nn.Linear(survey_dim, ehr_dim)
        )

        self.ehr_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)
        self.survey_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)
        self.output_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=ehr_dim,
            num_heads=attn_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        
    def forward(
        self,
        ehr_hidden,
        survey_hidden,
        patient_lengths,
        survey_attention_mask=None,
    ):
        """
        ehr_hidden:
            [sum(patient_lengths), ehr_dim]

        survey_hidden:
            [batch_size, survey_length, survey_dim]

        patient_lengths:
            [batch_size]

        survey_attention_mask:
            [batch_size, survey_length]
            1 = valid survey token
            0 = padding
        """

        # Keep a tensor on the same device as the EHR hidden states.
        patient_lengths_device = patient_lengths.to(
            device=ehr_hidden.device,
            dtype=torch.long,
        )

        # torch.split requires Python integers.
        patient_lengths_list = (
            patient_lengths_device.detach().cpu().tolist()
        )

        if sum(patient_lengths_list) != ehr_hidden.shape[0]:
            raise ValueError(
                "The sum of patient_lengths must equal the number "
                f"of packed EHR tokens. Got "
                f"{sum(patient_lengths_list)} and "
                f"{ehr_hidden.shape[0]}."
            )

        if survey_hidden.shape[0] != len(patient_lengths_list):
            raise ValueError(
                "The EHR and survey batch sizes do not match. "
                f"Got {len(patient_lengths_list)} EHR sequences "
                f"and {survey_hidden.shape[0]} survey sequences."
            )

        # Split packed EHR states into one sequence per patient.
        ehr_sequences = torch.split(
            ehr_hidden,
            patient_lengths_list,
            dim=0,
        )

        # [batch_size, max_ehr_length, ehr_dim]
        padded_ehr = pad_sequence(
            ehr_sequences,
            batch_first=True,
            padding_value=0.0,
        )

        max_ehr_length = padded_ehr.shape[1]

        # [batch_size, max_ehr_length]
        # True indicates a real EHR position.
        ehr_valid_mask = (
            torch.arange(
                max_ehr_length,
                device=ehr_hidden.device,
            ).unsqueeze(0)
            < patient_lengths_device.unsqueeze(1)
        )

        # [batch_size, survey_length, ehr_dim]
        survey_hidden = self.survey_embed_proj(survey_hidden)

        query = self.ehr_norm(padded_ehr)
        key_value = self.survey_norm(survey_hidden)

        survey_padding_mask = None

        if survey_attention_mask is not None:
            survey_attention_mask = survey_attention_mask.to(
                device=survey_hidden.device
            ).bool()

            if survey_attention_mask.shape != survey_hidden.shape[:2]:
                raise ValueError(
                    "survey_attention_mask must have shape "
                    "[batch_size, survey_length]."
                )

            # Every patient must have at least one survey token.
            if (~survey_attention_mask.any(dim=1)).any():
                raise ValueError(
                    "At least one patient has no valid survey tokens."
                )

            # MultiheadAttention:
            # True means ignore this key/value position.
            survey_padding_mask = ~survey_attention_mask

        # Batched cross-attention:
        #
        # Q = [B, max_EHR_length, ehr_dim]
        # K = [B, survey_length, ehr_dim]
        # V = [B, survey_length, ehr_dim]
        cross_output, _ = self.cross_attention(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=survey_padding_mask,
            need_weights=False,
        )

        # Residual connection around cross-attention.
        output = self.output_norm(
            padded_ehr + self.dropout(cross_output)
        )

        # Discard padded EHR query positions and restore FEMR's
        # packed ordering.
        packed_output = output[ehr_valid_mask]

        return packed_output

### Updated FEMR Encoder layer

In [ ]:
class FEMREncoderLayer(
    femr.models.transformer.FEMREncoderLayer
):
    def __init__(
        self,
        config,
        attn_heads,
        survey_dim,
        ehr_dim=None,
        dropout=0.1,
    ):
        super().__init__(config)

        if ehr_dim is None:
            ehr_dim = config.hidden_size

        if ehr_dim != config.hidden_size:
            raise ValueError(
                f"ehr_dim={ehr_dim} must match FEMR hidden size "
                f"{config.hidden_size}"
            )

        self.cross_attn = SurveyCrossAttn(
            attn_heads=attn_heads,
            survey_dim=survey_dim,
            ehr_dim=ehr_dim,
            dropout=dropout,
        )

    def forward(
        self,
        x,
        normed_ages,
        pos_embed,
        attn_bias,
        survey_hidden,
        patient_lengths,
        survey_attention_mask=None,
    ):
        # Original FEMR layer returns the update that is
        # normally added in FEMRTransformer.forward().
        x = super().forward(
            x=x,
            normed_ages=normed_ages,
            pos_embed=pos_embed,
            attn_bias=attn_bias,
        )

        # New survey cross-attention.
        
        # Q = EHR hidden states
        # K = survey hidden states
        # V = survey hidden states
        cross_attn = self.cross_attn(
            ehr_hidden=x,
            survey_hidden=survey_hidden,
            patient_lengths=patient_lengths,
            survey_attention_mask=survey_attention_mask,
        )

        return x + cross_attn # FEMR Residual connection

### Model Pipeline

In [ ]:
class GermLinePredModel(nn.Module):
    def __init__(
        self,
        survey_model,
        survey_tokenizer,
        ehr_model,
        ehr_tokenizer,
        survey_dim=None,
        ehr_dim=None,
        attn_heads=8,
        num_labels=2,
        dropout=0.1,
    ):
        super().__init__()

        self.survey_embed_model = survey_model
        self.survey_embed_token = survey_tokenizer

        self.ehr_model = ehr_model
        self.ehr_token = ehr_tokenizer

        ehr_transformer = self.ehr_model.transformer
        config = ehr_transformer.config

        if survey_dim is None:
            survey_dim = (
                self.survey_embed_model.config.hidden_size
            )

        if ehr_dim is None:
            ehr_dim = config.hidden_size

        if ehr_dim != config.hidden_size:
            raise ValueError(
                f"ehr_dim={ehr_dim} must match CLMBR hidden "
                f"size {config.hidden_size}"
            )

        # Replace every original FEMR layer with a layer that
        # contains both:
        #
        # 1. The original pretrained FEMR operation
        # 2. Survey cross-attention
        modified_layers = nn.ModuleList()

        for original_layer in ehr_transformer.layers:
            modified_layer = FEMREncoderLayer(
                config=config,
                attn_heads=attn_heads,
                survey_dim=survey_dim,
                ehr_dim=ehr_dim,
                dropout=dropout,
            )

            # Copy only the pretrained FEMR parameters:
            # norm, input_proj and output_proj.
            #
            # The cross-attention parameters remain newly initialized.
            modified_layer.load_state_dict(
                original_layer.state_dict(),
                strict=False,
            )

            modified_layers.append(modified_layer)

        ehr_transformer.layers = modified_layers

        self.classifier = nn.Sequential(
            nn.LayerNorm(ehr_dim),
            nn.Dropout(dropout),
            nn.Linear(ehr_dim, num_labels),
        )

    def forward(
        self,
        ehr_batch,
        survey_input_ids,
        survey_attention_mask,
        labels=None,
    ):
        # --------------------------------------------------
        # 1. Encode every patient's survey sequence
        # --------------------------------------------------

        survey_outputs = self.survey_embed_model(
            input_ids=survey_input_ids,
            attention_mask=survey_attention_mask,
            return_dict=True,
        )

        survey_hidden = survey_outputs.last_hidden_state

        # survey_hidden:
        # [batch_size, survey_length, survey_dim]

        # --------------------------------------------------
        # 2. Reproduce FEMRTransformer.forward()
        # --------------------------------------------------

        transformer = self.ehr_model.transformer
        config = transformer.config

        if not config.is_hierarchical:
            x = transformer.embed(
                ehr_batch["tokens"]
            )
        else:
            x = transformer.embed_bag(
                ehr_batch["hierarchical_tokens"],
                ehr_batch["token_indices"],
                ehr_batch["hierarchical_weights"],
            )

        # x is packed:
        # [sum(patient_lengths), ehr_dim]
        x = transformer.in_norm(x)

        normed_ages = ehr_batch["normalized_ages"]

        pos_embed = fixed_pos_embedding(
            ehr_batch["ages"],
            config.hidden_size // config.n_heads,
            x.dtype,
        )

        attn_bias = (
            xformers.ops.fmha.attn_bias.BlockDiagonalMask
            .from_seqlens(
                ehr_batch["patient_lengths"].tolist()
            )
            .make_local_attention(
                config.attention_width
            )
        )

        # --------------------------------------------------
        # 3. EHR self-attention + survey cross-attention
        #    in every CLMBR layer
        # --------------------------------------------------

        for layer in transformer.layers:
            x = layer(
                x=x,
                normed_ages=normed_ages,
                pos_embed=pos_embed,
                attn_bias=attn_bias,
                survey_hidden=survey_hidden,
                patient_lengths=(
                    ehr_batch["patient_lengths"]
                ),
                survey_attention_mask=(
                    survey_attention_mask
                ),
            )

        x = transformer.out_norm(x)

        # --------------------------------------------------
        # 4. Select one representation per patient
        # --------------------------------------------------

        patient_lengths = ehr_batch["patient_lengths"]

        # Because x is packed, the final event for each patient
        # is located at cumulative_length - 1.
        final_event_indices = (
            torch.cumsum(patient_lengths, dim=0) - 1
        ).long()

        patient_embeddings = x[final_event_indices]

        # patient_embeddings:
        # [batch_size, ehr_dim]

        logits = self.classifier(
            patient_embeddings
        )

        loss = None

        if labels is not None:
            if logits.shape[-1] == 1:
                loss = F.binary_cross_entropy_with_logits(
                    logits.squeeze(-1),
                    labels.float(),
                )
            else:
                loss = F.cross_entropy(
                    logits,
                    labels.long(),
                )

        return {
            "loss": loss,
            "logits": logits,
            "patient_embeddings": patient_embeddings,
        }
        
        

# Load Data

In [ ]:
genetic_df = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv')
survey_df = pd.read_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey.parquet')

In [ ]:
survey_df.head()

In [ ]:
from pathlib import Path

dir_path = Path('/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/data/')

In [ ]:
meds_patients = set()

for file in dir_path.glob("*/*.parquet"):
    meds = pd.read_parquet(file)
    patients = meds.subject_id.unique()
    meds_patients.update(patients)
shared_patients = set(survey_df["person_id"]) & meds_patients

In [ ]:
len(shared_patients)

In [ ]:
entrie_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

In [ ]:
entrie_v8.s.nunique()

In [ ]:
for file in dir_path.glob("*/*.parquet"):
    meds = pd.read_parquet(file)
    meds = meds[meds['subject_id'].isin(shared_patients)]
    meds.to_parquet(file, index=False)

survey_df = survey_df[survey_df['person_id'].isin(shared_patients)]
survey_df.to_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey_filt.parquet')
genetic_df = genetic_df[genetic_df['s'].isin(shared_patients)]
genetic_df.to_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v9.csv')